# Notebook 2: Descriptive Analysis

**Project:** How Time and Location Choices Change the Carbon Footprint of AI Training  
**Data:** EIA Form 930 BALANCE — Jan–Dec 2025  
**Author:** Lin Htet Aung — MSc Business Analytics, Trinity College Dublin

### Research Questions Addressed
- **RQ1:** How large are the differences in hourly carbon exposure across U.S. regional grid areas?
- **RQ2:** Within the same region, how strongly does carbon exposure vary by hour of day and day of week?

---
**Prerequisites:** Run `01_data_prep.ipynb` first, which produces `data/processed/analysis_dataset.csv`

## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)

print('Libraries loaded successfully.')

## 1. Load Data

In [ ]:
FILE_PATH = '../data/processed/analysis_dataset.csv'
df_raw = pd.read_csv(FILE_PATH, low_memory=False)

print(f'Raw data loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
print(f'Months covered: {sorted(df_raw["month"].unique())}')
print(f'Regions: {sorted(df_raw["region"].dropna().unique())}')
print(f'Balancing Authorities: {df_raw["ba"].nunique()}')

## 2. Data Cleaning & Variable Construction

In [ ]:
df = df_raw.copy()

# Standardise column names
df = df.rename(columns={
    'ba':           'Balancing Authority',
    'region':       'Region',
    'demand_mw':    'Demand (MW) (Adjusted)',
    'net_gen_mw':   'Net Generation (MW) (Adjusted)',
    'gen_coal_mw':  'Net Generation (MW) from Coal (Adjusted)',
    'gen_gas_mw':   'Net Generation (MW) from Natural Gas (Adjusted)',
    'gen_petro_mw': 'Net Generation (MW) from All Petroleum Products (Adjusted)',
})

COAL_COL    = 'Net Generation (MW) from Coal (Adjusted)'
GAS_COL     = 'Net Generation (MW) from Natural Gas (Adjusted)'
PETRO_COL   = 'Net Generation (MW) from All Petroleum Products (Adjusted)'
NET_GEN_COL = 'Net Generation (MW) (Adjusted)'
DEMAND_COL  = 'Demand (MW) (Adjusted)'

In [ ]:
# Parse datetime & create time features
df['datetime'] = pd.to_datetime(df['local_datetime'])
df['month']       = df['datetime'].dt.month
df['month_name']  = df['datetime'].dt.strftime('%b')
df['day_of_week'] = df['datetime'].dt.dayofweek   # 0=Mon, 6=Sun
df['day_name']    = df['datetime'].dt.strftime('%a')
df['hour']        = df['datetime'].dt.hour
df['is_weekend']  = df['day_of_week'].isin([5, 6]).astype(int)

df['season'] = df['month'].map({
    12: 'Winter', 1: 'Winter',  2: 'Winter',
     3: 'Spring', 4: 'Spring',  5: 'Spring',
     6: 'Summer', 7: 'Summer',  8: 'Summer',
     9: 'Fall',  10: 'Fall',   11: 'Fall'
})

print(df[['datetime', 'month', 'hour', 'day_name', 'is_weekend', 'season']].head(3))

In [ ]:
# Handle missing values in fossil fuel columns
# NaN = fuel type absent from BA reporting structure (genuine zero, not missing)
for col in [COAL_COL, GAS_COL, PETRO_COL]:
    if col in df.columns:
        df[col] = df[col].fillna(0)

before = len(df)
df = df.dropna(subset=[NET_GEN_COL])
print(f'Rows dropped (missing net generation): {before - len(df):,}')
print(f'Remaining rows: {len(df):,}')

In [ ]:
# Construct carbon exposure proxy: fossil share of net generation
# Carbon Proxy(r,t) = (Coal + Gas + Petroleum)(r,t) / Net Generation(r,t)

if 'carbon_proxy' in df.columns:
    df['carbon_proxy'] = df['carbon_proxy'].clip(0, 1)
    print('Using pre-computed carbon_proxy column.')
else:
    df['fossil_total'] = df[COAL_COL] + df[GAS_COL] + df[PETRO_COL]
    df['carbon_proxy'] = np.where(
        df[NET_GEN_COL] > 0,
        df['fossil_total'] / df[NET_GEN_COL],
        np.nan
    )
    df['carbon_proxy'] = df['carbon_proxy'].clip(0, 1)

df = df.dropna(subset=['carbon_proxy'])
print(f'Final dataset: {len(df):,} BA-hour observations')
print(f'Date range: {df["datetime"].min()} → {df["datetime"].max()}')

## 3. Descriptive Analysis

### 3.1 Overview Statistics

In [ ]:
print('=== Carbon Exposure Proxy — Summary Statistics ===')
print(df['carbon_proxy'].describe().round(4))

print('\n=== By Region ===')
region_stats = df.groupby('Region')['carbon_proxy'].agg(
    ['mean', 'median', 'std', 'min', 'max']
).round(3)
region_stats.columns = ['Mean', 'Median', 'Std Dev', 'Min', 'Max']
region_stats = region_stats.sort_values('Mean', ascending=False)
print(region_stats.to_string())

### 3.2 RQ1 — Cross-Regional Differences in Carbon Exposure

In [ ]:
region_order = region_stats.index.tolist()
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ci = df.groupby('Region')['carbon_proxy'].apply(
    lambda x: stats.sem(x, ddof=1) * stats.t.ppf(0.975, len(x) - 1)
)
means = df.groupby('Region')['carbon_proxy'].mean().reindex(region_order)
errs  = ci.reindex(region_order)

axes[0].barh(region_order[::-1], means[::-1], xerr=errs[::-1],
             color=sns.color_palette('RdYlGn_r', len(region_order)),
             edgecolor='grey', linewidth=0.5, capsize=4)
axes[0].set_xlabel('Mean Carbon Exposure Proxy (Fossil Share)')
axes[0].set_title('RQ1: Average Carbon Exposure by U.S. Region\n(Jan–Dec 2025, 95% CI)')
axes[0].axvline(means.mean(), color='black', linestyle='--', linewidth=1, label='Overall mean')
axes[0].legend()
for i, (v, e) in enumerate(zip(means[::-1], errs[::-1])):
    axes[0].text(v + e + 0.005, i, f'{v:.3f}', va='center', fontsize=9)

plot_data = [df.loc[df['Region'] == r, 'carbon_proxy'].dropna().values for r in region_order]
bp = axes[1].boxplot(plot_data, vert=False, patch_artist=True, notch=True,
                     labels=region_order,
                     medianprops=dict(color='black', linewidth=2))
colors = sns.color_palette('RdYlGn_r', len(region_order))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
axes[1].set_xlabel('Carbon Exposure Proxy (Fossil Share)')
axes[1].set_title('RQ1: Distribution of Carbon Exposure by Region\n(notched box = 95% CI around median)')

plt.tight_layout()
plt.savefig('../outputs/figures/fig1_region_comparison.png', bbox_inches='tight')
plt.show()
print('Figure saved.')

In [ ]:
# One-way ANOVA: confirm regional differences are statistically significant
groups = [df.loc[df['Region'] == r, 'carbon_proxy'].dropna().values for r in region_order]
f_stat, p_val = stats.f_oneway(*groups)
print(f'One-way ANOVA across regions:')
print(f'  F-statistic = {f_stat:,.1f},  p-value = {p_val:.4e}')

highest = region_stats['Mean'].idxmax()
lowest  = region_stats['Mean'].idxmin()
spread  = region_stats['Mean'].max() - region_stats['Mean'].min()
print(f'Highest mean: {highest} ({region_stats.loc[highest, "Mean"]:.3f})')
print(f'Lowest  mean: {lowest}  ({region_stats.loc[lowest,  "Mean"]:.3f})')
print(f'Cross-regional spread: {spread:.3f} ({spread*100:.1f} pp)')

### 3.3 BA-Level Detail Within Regions

In [ ]:
ba_region = df.groupby(['Balancing Authority', 'Region'])['carbon_proxy'].mean().reset_index()
ba_region = ba_region.sort_values('carbon_proxy', ascending=False)

fig, ax = plt.subplots(figsize=(14, 9))
palette = {
    r: c for r, c in zip(
        sorted(ba_region['Region'].unique()),
        sns.color_palette('tab10', ba_region['Region'].nunique())
    )
}
colors_ba = ba_region['Region'].map(palette)
ax.barh(ba_region['Balancing Authority'], ba_region['carbon_proxy'],
        color=colors_ba, edgecolor='white', linewidth=0.4)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=palette[r], label=r) for r in sorted(palette)]
ax.legend(handles=legend_elements, title='Region', loc='lower right')
ax.set_xlabel('Mean Carbon Exposure Proxy (Fossil Share)')
ax.set_title('Carbon Exposure by Balancing Authority (BA)\nColoured by Region — Jan–Dec 2025')
ax.axvline(ba_region['carbon_proxy'].mean(), color='red', linestyle='--', linewidth=1.2)

plt.tight_layout()
plt.savefig('../outputs/figures/fig2_ba_comparison.png', bbox_inches='tight')
plt.show()

### 3.4 RQ2 — Within-Region Variation by Hour of Day

In [ ]:
hourly_region = df.groupby(['Region', 'hour'])['carbon_proxy'].mean().reset_index()
palette_r = sns.color_palette('tab10', df['Region'].nunique())

fig, ax = plt.subplots(figsize=(14, 6))
for (r, color) in zip(sorted(df['Region'].unique()), palette_r):
    sub = hourly_region[hourly_region['Region'] == r].sort_values('hour')
    ax.plot(sub['hour'], sub['carbon_proxy'], marker='o', markersize=4,
            linewidth=2, label=r, color=color)

ax.set_xticks(range(0, 24))
ax.set_xticklabels([f'{h:02d}:00' for h in range(24)], rotation=45, ha='right', fontsize=8)
ax.set_xlabel('Hour of Day (Local)')
ax.set_ylabel('Mean Carbon Exposure Proxy')
ax.set_title('RQ2: Hourly Carbon Exposure Pattern by Region (Jan–Dec 2025)')
ax.legend(title='Region', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.axvspan(22, 23.99, alpha=0.06, color='navy')
ax.axvspan(0, 6, alpha=0.06, color='navy')

plt.tight_layout()
plt.savefig('../outputs/figures/fig3_hourly_by_region.png', bbox_inches='tight')
plt.show()

# Quantify within-region hourly spread
hourly_spread = hourly_region.groupby('Region')['carbon_proxy'].apply(
    lambda x: x.max() - x.min()
).sort_values(ascending=False)

print('Within-region hourly spread (max hourly mean − min hourly mean):')
for r, v in hourly_spread.items():
    print(f'  {r}: {v:.4f}  ({v*100:.1f} percentage points)')

### 3.5 RQ2 — Weekday vs Weekend Pattern

In [ ]:
wd_we = df.groupby(['Region', 'hour', 'is_weekend'])['carbon_proxy'].mean().reset_index()
regions_sorted = region_stats.index.tolist()

n = len(regions_sorted)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 4), sharey=False)
axes = axes.flatten()

for i, r in enumerate(regions_sorted):
    sub = wd_we[wd_we['Region'] == r].sort_values('hour')
    wd  = sub[sub['is_weekend'] == 0]
    we  = sub[sub['is_weekend'] == 1]
    axes[i].plot(wd['hour'], wd['carbon_proxy'], label='Weekday', color='steelblue', linewidth=2)
    axes[i].plot(we['hour'], we['carbon_proxy'], label='Weekend', color='tomato', linewidth=2, linestyle='--')
    axes[i].set_title(f'Region: {r}')
    axes[i].set_xlabel('Hour of Day')
    axes[i].set_ylabel('Mean Carbon Proxy')
    axes[i].legend(fontsize=8)
    axes[i].set_xticks(range(0, 24, 4))

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('RQ2: Hourly Carbon Exposure — Weekday vs Weekend by Region', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../outputs/figures/fig4_weekday_weekend.png', bbox_inches='tight')
plt.show()

# Welch's t-tests
print('T-test (Weekday vs Weekend) by Region:')
print(f'{"Region":<8} {"WD Mean":>10} {"WE Mean":>10} {"Diff":>8} {"p-value":>12}')
for r in regions_sorted:
    wd_vals = df[(df['Region'] == r) & (df['is_weekend'] == 0)]['carbon_proxy']
    we_vals = df[(df['Region'] == r) & (df['is_weekend'] == 1)]['carbon_proxy']
    t, p = stats.ttest_ind(wd_vals, we_vals, equal_var=False)
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    print(f'{r:<8} {wd_vals.mean():>10.4f} {we_vals.mean():>10.4f} '
          f'{(wd_vals.mean() - we_vals.mean()):>8.4f} {p:>12.4e} {sig}')

### 3.6 Monthly / Seasonal Patterns

In [ ]:
month_order  = list(range(1, 13))
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_region = df.groupby(['Region', 'month'])['carbon_proxy'].mean().reset_index()

fig, ax = plt.subplots(figsize=(14, 6))
for r, color in zip(sorted(df['Region'].unique()), palette_r):
    sub = monthly_region[monthly_region['Region'] == r].sort_values('month')
    ax.plot(sub['month'], sub['carbon_proxy'], marker='s', markersize=6,
            linewidth=2, label=r, color=color)

ax.set_xticks(month_order)
ax.set_xticklabels(month_labels)
ax.set_xlabel('Month')
ax.set_ylabel('Mean Carbon Exposure Proxy')
ax.set_title('Seasonal Patterns in Carbon Exposure by Region (Jan–Dec 2025)')
ax.legend(title='Region', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.axvspan(0.5, 3.5, alpha=0.07, color='lightblue')
ax.axvspan(3.5, 6.5, alpha=0.07, color='lightgreen')
ax.axvspan(6.5, 9.5, alpha=0.07, color='gold')
ax.axvspan(9.5, 12.5, alpha=0.07, color='orange')

plt.tight_layout()
plt.savefig('../outputs/figures/fig5_monthly_seasonal.png', bbox_inches='tight')
plt.show()

### 3.7 Carbon Exposure vs Demand: Are High-Demand Hours More Carbon-Intensive?

In [ ]:
df_demand = df.dropna(subset=[DEMAND_COL]).copy()
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sample = df_demand.sample(min(30000, len(df_demand)), random_state=42)
for r, color in zip(sorted(sample['Region'].unique()), palette_r):
    sub = sample[sample['Region'] == r]
    axes[0].scatter(sub[DEMAND_COL], sub['carbon_proxy'],
                    alpha=0.15, s=5, color=color, label=r)
axes[0].set_xlabel('Electricity Demand (MW)')
axes[0].set_ylabel('Carbon Exposure Proxy')
axes[0].set_title('Carbon Exposure vs Demand (sample 30k points)')
axes[0].legend(title='Region', markerscale=3, fontsize=8)

corr_data = (
    df_demand.groupby('Region')
    .apply(lambda g: g[['carbon_proxy', DEMAND_COL]].corr().iloc[0, 1])
    .reset_index()
)
corr_data.columns = ['Region', 'Pearson_r']
corr_data = corr_data.sort_values('Pearson_r')
bar_colors = ['tomato' if v > 0 else 'steelblue' for v in corr_data['Pearson_r']]
axes[1].barh(corr_data['Region'], corr_data['Pearson_r'], color=bar_colors, edgecolor='grey')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Pearson Correlation (r)')
axes[1].set_title('Correlation: Demand vs Carbon Exposure by Region')
for i, (v, r) in enumerate(zip(corr_data['Pearson_r'], corr_data['Region'])):
    axes[1].text(v + (0.005 if v >= 0 else -0.005), i, f'{v:.3f}', va='center',
                 ha='left' if v >= 0 else 'right', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/figures/fig6_demand_vs_carbon.png', bbox_inches='tight')
plt.show()
print(corr_data.to_string(index=False))

### 3.8 Heatmap: Hour × Day-of-Week Carbon Exposure (All Regions)

In [ ]:
day_abbr = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
n = len(regions_sorted)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 5))
axes = axes.flatten()

vmin = df.groupby(['hour', 'day_of_week'])['carbon_proxy'].mean().min()
vmax = df.groupby(['hour', 'day_of_week'])['carbon_proxy'].mean().max()

for i, r in enumerate(regions_sorted):
    pivot = (
        df[df['Region'] == r]
        .groupby(['hour', 'day_of_week'])['carbon_proxy']
        .mean()
        .unstack('day_of_week')
    )
    pivot.columns = day_abbr
    sns.heatmap(pivot, ax=axes[i], cmap='RdYlGn_r', vmin=vmin, vmax=vmax,
                linewidths=0.3, annot=False, cbar=True,
                cbar_kws={'label': 'Carbon Proxy'})
    axes[i].set_title(f'Region: {r}', fontsize=11)
    axes[i].set_xlabel('Day of Week')
    axes[i].set_ylabel('Hour of Day')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Carbon Exposure Heatmap: Hour × Day of Week by Region', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../outputs/figures/fig7_heatmaps.png', bbox_inches='tight')
plt.show()

## 4. Scenario Simulation — AI Training Workload Scheduling

Simulates 4 scheduling strategies for a fixed AI training workload:
- **Baseline:** Execute immediately at default region
- **Time-shift:** Defer within same region to lowest-carbon window  
- **Location-shift:** Route to lowest-carbon region at same time  
- **Joint optimisation:** Defer AND route to minimise exposure

Workload assumptions: 4 hours continuous, flexibility window ±6h / ±12h / ±24h

In [ ]:
# Build region-hour lookup table (mean observed proxy values)
df_sim = df.copy()
df_sim['month_num'] = df_sim['month']
lookup = (
    df_sim
    .groupby(['Region', 'month_num', 'day_of_week', 'hour'])['carbon_proxy']
    .mean()
    .reset_index()
    .rename(columns={'carbon_proxy': 'mean_proxy'})
)
print(f'Lookup table rows: {len(lookup):,}')

def get_proxy(region, month, dow, hour):
    row = lookup[
        (lookup['Region'] == region) &
        (lookup['month_num'] == month) &
        (lookup['day_of_week'] == dow) &
        (lookup['hour'] == hour)
    ]
    return row.iloc[0]['mean_proxy'] if len(row) > 0 else np.nan

def simulate_workload(base_region, base_month, base_dow, base_hour,
                      duration=4, flex_hours=12):
    all_regions = sorted(df['Region'].unique())

    def avg_block(region, month, dow, start_hour, dur=duration):
        vals = [get_proxy(region, month, dow, (start_hour + i) % 24) for i in range(dur)]
        vals = [v for v in vals if not np.isnan(v)]
        return np.mean(vals) if vals else np.nan

    baseline = avg_block(base_region, base_month, base_dow, base_hour)

    # Time-shifting
    best_time, best_time_hour = baseline, base_hour
    for delta in range(-flex_hours, flex_hours + 1):
        h = (base_hour + delta) % 24
        v = avg_block(base_region, base_month, base_dow, h)
        if not np.isnan(v) and v < best_time:
            best_time, best_time_hour = v, h

    # Location-shifting
    best_loc, best_loc_region = baseline, base_region
    for r in all_regions:
        v = avg_block(r, base_month, base_dow, base_hour)
        if not np.isnan(v) and v < best_loc:
            best_loc, best_loc_region = v, r

    # Joint optimisation
    best_joint, best_joint_region, best_joint_hour = baseline, base_region, base_hour
    for r in all_regions:
        for delta in range(-flex_hours, flex_hours + 1):
            h = (base_hour + delta) % 24
            v = avg_block(r, base_month, base_dow, h)
            if not np.isnan(v) and v < best_joint:
                best_joint, best_joint_region, best_joint_hour = v, r, h

    return {
        'baseline':   {'proxy': baseline,   'region': base_region,       'hour': base_hour},
        'time_shift': {'proxy': best_time,   'region': base_region,       'hour': best_time_hour},
        'loc_shift':  {'proxy': best_loc,    'region': best_loc_region,   'hour': base_hour},
        'joint':      {'proxy': best_joint,  'region': best_joint_region, 'hour': best_joint_hour},
    }

print('Simulation function defined.')

In [ ]:
# Run simulation: Monday 14:00 (peak afternoon), September, 4h workload
all_regions = sorted(df['Region'].unique())
rows = []
for flex_h in [6, 12, 24]:
    for base_r in all_regions:
        res = simulate_workload(
            base_region=base_r, base_month=9,
            base_dow=0, base_hour=14,
            duration=4, flex_hours=flex_h
        )
        rows.append({
            'Base Region': base_r, 'Flex Window (h)': flex_h,
            'Baseline': res['baseline']['proxy'],
            'Time Shift': res['time_shift']['proxy'],
            'Loc Shift': res['loc_shift']['proxy'],
            'Joint': res['joint']['proxy'],
            'Best Time Hour': res['time_shift']['hour'],
            'Best Loc Region': res['loc_shift']['region'],
            'Best Joint Region': res['joint']['region'],
            'Best Joint Hour': res['joint']['hour'],
        })

sim_df = pd.DataFrame(rows)
for col in ['Time Shift', 'Loc Shift', 'Joint']:
    sim_df[f'{col} Reduction %'] = (
        (sim_df['Baseline'] - sim_df[col]) / sim_df['Baseline'] * 100
    )

display_cols = ['Base Region', 'Baseline', 'Time Shift', 'Loc Shift', 'Joint',
                'Time Shift Reduction %', 'Loc Shift Reduction %', 'Joint Reduction %']
print('=== Simulation Results (flex=12h, Sep Monday 14:00, 4h workload) ===')
print(sim_df[sim_df['Flex Window (h)'] == 12][display_cols].to_string(
    index=False, float_format='{:.4f}'.format))

In [ ]:
# Chart: Average reduction by strategy and flexibility window
agg = (
    sim_df.groupby('Flex Window (h)')[
        ['Time Shift Reduction %', 'Loc Shift Reduction %', 'Joint Reduction %']
    ].mean()
)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(agg))
width = 0.25
ax.bar(x - width, agg['Time Shift Reduction %'],  width, label='Time-Shifting',      color='steelblue')
ax.bar(x,          agg['Loc Shift Reduction %'],   width, label='Location-Shifting',  color='tomato')
ax.bar(x + width,  agg['Joint Reduction %'],       width, label='Joint Optimisation', color='forestgreen')
ax.set_xticks(x)
ax.set_xticklabels([f'±{h}h' for h in agg.index])
ax.set_xlabel('Flexibility Window')
ax.set_ylabel('Average Carbon Exposure Reduction (%)')
ax.set_title('Simulation: Average Reduction vs Baseline\nby Scheduling Strategy and Flexibility Window')
ax.legend()
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', fontsize=8, padding=2)

plt.tight_layout()
plt.savefig('../outputs/figures/fig10_simulation_strategy.png', bbox_inches='tight')
plt.show()
print(agg.round(2))